In [1]:
!pip install qdrant-client sentence-transformers rank-bm25 groq pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.4/390.4 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 9.9 MB/s eta 0:00:00


In [14]:
from google.colab import userdata

groq_api_key = userdata.get('GROQ_API_KEY')
qdrant_api_key = userdata.get('QDRANT_API_KEY')
qdrant_url = userdata.get('QDRANT_URL')

print("All keys loaded successfully")

All keys loaded successfully


In [15]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, SparseVectorParams, SparseIndexParams

client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)
print("Qdrant connected")
print("Collections:", client.get_collections())

Qdrant connected
Collections: collections=[CollectionDescription(name='resume_matcher')]


In [16]:
from sentence_transformers import SentenceTransformer

print("Loading BGE model...")
embedding_model = SentenceTransformer("BAAI/bge-large-en-v1.5")
print("BGE model loaded")

test = embedding_model.encode("test")
print("Embedding dimension:", len(test))

Loading BGE model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BGE model loaded
Embedding dimension: 1024


In [17]:
import re
from rank_bm25 import BM25Okapi

def text_to_tokens(text: str) -> list:
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    tokens = [t for t in text.split() if len(t) > 1]
    return tokens

# Real resume corpus - this will grow as more resumes are added
resume_corpus = [
    "Python Machine Learning Deep Learning NLP TensorFlow PyTorch SQL Docker Git FastAPI Scikit-learn Pandas NumPy BERT text classification deployment AWS EC2 data analysis visualization",
    "Java Spring Boot Microservices Kubernetes AWS Docker Jenkins CICD REST APIs backend software engineer infrastructure cloud",
    "Python NLP Transformers BERT LangChain RAG LLM HuggingFace Vector Databases Qdrant ChromaDB embeddings semantic search fine-tuning",
    "Python Data Engineering Spark Hadoop Airflow ETL Pipeline SQL PostgreSQL AWS S3 data warehouse analytics Flipkart",
    "Python TensorFlow Computer Vision OpenCV YOLO Object Detection Image Classification CNN ResNet deep learning research",
]

# Build vocabulary dynamically from corpus
all_tokens = []
for doc in resume_corpus:
    all_tokens.extend(text_to_tokens(doc))

# Remove duplicates and build vocabulary
unique_tokens = list(set(all_tokens))
vocabulary = {token: idx for idx, token in enumerate(unique_tokens)}

# Build BM25 index from corpus
tokenized_corpus = [text_to_tokens(doc) for doc in resume_corpus]
bm25_index = BM25Okapi(tokenized_corpus)

print("Vocabulary built dynamically from resume corpus")
print("Vocabulary size:", len(vocabulary))
print("BM25 index built successfully")
print("Sample vocabulary:", list(vocabulary.keys())[:10])

Vocabulary built dynamically from resume corpus
Vocabulary size: 73
BM25 index built successfully
Sample vocabulary: ['search', 'langchain', 'fine', 'semantic', 'git', 'bert', 'boot', 'airflow', 'opencv', 'vision']


In [18]:
from qdrant_client.models import SparseVector

def build_sparse_vector(text: str) -> SparseVector:
    tokens = text_to_tokens(text)

    # Calculate term frequency from vocabulary
    tf = {}
    for token in tokens:
        if token in vocabulary:
            idx = vocabulary[token]
            tf[idx] = tf.get(idx, 0) + 1

    if not tf:
        return SparseVector(indices=[0], values=[0.0])

    # Normalize by document length
    max_freq = max(tf.values())
    indices = list(tf.keys())
    values = [round(float(v) / max_freq, 4) for v in tf.values()]

    return SparseVector(indices=indices, values=values)

# Test it
test_sparse = build_sparse_vector("Python Machine Learning NLP Docker")
print("Sparse vector working correctly")
print("Non-zero indices:", len(test_sparse.indices))
print("Sample indices:", test_sparse.indices[:5])
print("Sample values:", test_sparse.values[:5])

Sparse vector working correctly
Non-zero indices: 5
Sample indices: [65, 35, 18, 47, 50]
Sample values: [1.0, 1.0, 1.0, 1.0, 1.0]


In [19]:
from pydantic import BaseModel
from typing import List, Optional, Any

# Same schemas as Notebook 1
class Experience(BaseModel):
    role: str
    company: str
    duration: str
    description: str

class Education(BaseModel):
    degree: str
    institution: str
    year: Optional[Any] = None

class ParsedResume(BaseModel):
    name: str
    email: Optional[str] = None
    phone: Optional[str] = None
    skills: List[str]
    experience: List[Experience]
    education: List[Education]
    certifications: List[str] = []
    total_experience_years: Optional[float] = None

class ParsedJD(BaseModel):
    job_title: str
    company: Optional[str] = None
    required_skills: List[str]
    preferred_skills: List[str] = []
    minimum_experience_years: Optional[float] = None
    education_requirement: Optional[str] = None
    certifications_required: List[str] = []
    responsibilities: List[str] = []
    location: Optional[str] = None

# Real parsed resumes - exactly as Notebook 1 would output
parsed_resumes = [
    ParsedResume(
        name="Charan Kumar",
        email="charan@gmail.com",
        phone="+91-9876543210",
        skills=["Python", "Machine Learning", "Deep Learning", "NLP",
                "TensorFlow", "PyTorch", "SQL", "Docker", "Git",
                "FastAPI", "Scikit-learn", "Pandas", "NumPy"],
        experience=[
            Experience(role="Machine Learning Intern", company="TechStartup Pvt Ltd",
                      duration="June 2024 - December 2024",
                      description="Built text classification models using BERT. Deployed ML models using FastAPI and Docker on AWS EC2."),
            Experience(role="Data Science Intern", company="Analytics Corp",
                      duration="January 2024 - May 2024",
                      description="EDA on sales datasets using Pandas. Created dashboards using Matplotlib.")
        ],
        education=[Education(degree="BTech Computer Science", institution="IIIT Kota", year=2025)],
        certifications=["Deep Learning Specialization Coursera", "AWS Cloud Practitioner"],
        total_experience_years=0.5
    ),
    ParsedResume(
        name="Rahul Verma",
        email="rahul@gmail.com",
        phone="+91-9123456789",
        skills=["Python", "NLP", "Transformers", "BERT", "LangChain",
                "RAG", "LLM", "HuggingFace", "Machine Learning",
                "Deep Learning", "Vector Databases", "Qdrant"],
        experience=[
            Experience(role="NLP Engineer", company="AI Startup",
                      duration="January 2024 - Present",
                      description="Built RAG systems using LangChain and Qdrant. Fine-tuned LLMs using HuggingFace."),
            Experience(role="Research Intern", company="IIT Delhi",
                      duration="June 2023 - December 2023",
                      description="Research on transformer architectures for NLP tasks.")
        ],
        education=[Education(degree="MTech Artificial Intelligence", institution="IIT Bombay", year=2024)],
        certifications=["HuggingFace NLP Course"],
        total_experience_years=1.5
    ),
    ParsedResume(
        name="Priya Sharma",
        email="priya@gmail.com",
        phone="+91-9988776655",
        skills=["Java", "Spring Boot", "Microservices", "Kubernetes",
                "AWS", "Docker", "Jenkins", "REST APIs", "MySQL"],
        experience=[
            Experience(role="Software Engineer", company="Infosys",
                      duration="July 2023 - Present",
                      description="Built microservices using Spring Boot. Deployed on Kubernetes and AWS."),
            Experience(role="Backend Developer", company="TCS",
                      duration="January 2022 - June 2023",
                      description="Developed REST APIs using Java Spring Boot.")
        ],
        education=[Education(degree="BTech Information Technology", institution="NIT Jaipur", year=2022)],
        certifications=[],
        total_experience_years=2.0
    ),
    ParsedResume(
        name="Sneha Patel",
        email="sneha@gmail.com",
        phone="+91-9876512345",
        skills=["Python", "Apache Spark", "Hadoop", "Airflow",
                "ETL", "SQL", "PostgreSQL", "AWS S3", "Data Engineering"],
        experience=[
            Experience(role="Data Engineer", company="Flipkart",
                      duration="August 2022 - Present",
                      description="Built ETL pipelines using Spark and Airflow. Managed data warehouse on AWS."),
            Experience(role="Analytics Engineer", company="Swiggy",
                      duration="January 2021 - July 2022",
                      description="Built data pipelines and analytics dashboards.")
        ],
        education=[Education(degree="BTech CSE", institution="BITS Pilani", year=2021)],
        certifications=["AWS Data Engineer Associate"],
        total_experience_years=3.0
    ),
    ParsedResume(
        name="Arjun Mehta",
        email="arjun@gmail.com",
        phone="+91-9765432109",
        skills=["Python", "TensorFlow", "Computer Vision", "OpenCV",
                "YOLO", "Object Detection", "CNN", "Deep Learning",
                "Image Classification", "PyTorch"],
        experience=[
            Experience(role="Computer Vision Engineer", company="Ola",
                      duration="March 2023 - Present",
                      description="Built object detection models using YOLO. Deployed CV pipelines in production."),
            Experience(role="Research Assistant", company="IISc",
                      duration="August 2022 - February 2023",
                      description="Research on CNN architectures for image recognition.")
        ],
        education=[Education(degree="MTech ECE", institution="IISc Bangalore", year=2023)],
        certifications=[],
        total_experience_years=2.0
    )
]

# Real parsed JD - exactly as Notebook 1 would output
parsed_jd = ParsedJD(
    job_title="Machine Learning Engineer",
    company="Google India",
    required_skills=["Python", "Machine Learning", "Deep Learning", "NLP", "Docker", "Git"],
    preferred_skills=["LangChain", "HuggingFace", "RAG", "MLflow", "Kubernetes"],
    minimum_experience_years=1.0,
    education_requirement="BTech or MTech in Computer Science",
    certifications_required=[],
    responsibilities=[
        "Build and deploy production ML models",
        "Design and implement NLP pipelines",
        "Monitor model performance in production"
    ],
    location="Bangalore, India"
)

print("All resumes loaded as proper Pydantic objects")
print("JD loaded as proper Pydantic object")
print()
print("JD:", parsed_jd.job_title, "at", parsed_jd.company)
print("Required Skills:", parsed_jd.required_skills)
print("Preferred Skills:", parsed_jd.preferred_skills)
print("Min Experience:", parsed_jd.minimum_experience_years, "years")
print()
print("Candidates loaded:")
for r in parsed_resumes:
    print(f"  - {r.name}: {r.total_experience_years} years, {len(r.skills)} skills")

All resumes loaded as proper Pydantic objects
JD loaded as proper Pydantic object

JD: Machine Learning Engineer at Google India
Required Skills: ['Python', 'Machine Learning', 'Deep Learning', 'NLP', 'Docker', 'Git']
Preferred Skills: ['LangChain', 'HuggingFace', 'RAG', 'MLflow', 'Kubernetes']
Min Experience: 1.0 years

Candidates loaded:
  - Charan Kumar: 0.5 years, 13 skills
  - Rahul Verma: 1.5 years, 12 skills
  - Priya Sharma: 2.0 years, 9 skills
  - Sneha Patel: 3.0 years, 9 skills
  - Arjun Mehta: 2.0 years, 10 skills


In [20]:
from qdrant_client.models import PointStruct, VectorParams, SparseVectorParams, SparseIndexParams, Distance

COLLECTION_NAME = "resume_matcher"

# Fresh start
try:
    client.delete_collection(COLLECTION_NAME)
    print("Old collection deleted")
except:
    pass

# Create collection with dense + sparse vectors
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense": VectorParams(size=1024, distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams(
            index=SparseIndexParams(on_disk=False)
        )
    }
)
print("Collection created with dense + sparse vectors")

# Index all resumes
points = []
for idx, resume in enumerate(parsed_resumes):

    # Build rich text for embedding - combines all resume fields
    resume_text = f"""
    Name: {resume.name}
    Skills: {' '.join(resume.skills)}
    Experience: {' '.join([f"{e.role} at {e.company}: {e.description}" for e in resume.experience])}
    Education: {' '.join([f"{e.degree} from {e.institution}" for e in resume.education])}
    Certifications: {' '.join(resume.certifications)}
    Total Experience: {resume.total_experience_years} years
    """

    # Dense vector
    dense_vector = embedding_model.encode(
        resume_text.strip(),
        normalize_embeddings=True
    ).tolist()

    # Sparse vector
    sparse_vector = build_sparse_vector(resume_text)

    point = PointStruct(
        id=idx + 1,
        vector={
            "dense": dense_vector,
            "sparse": sparse_vector
        },
        payload={
            "name": resume.name,
            "email": resume.email,
            "skills": resume.skills,
            "certifications": resume.certifications,
            "exp_years": resume.total_experience_years,
            "education": resume.education[0].degree,
            "institution": resume.education[0].institution,
            "text": resume_text.strip()
        }
    )
    points.append(point)

client.upsert(collection_name=COLLECTION_NAME, points=points)

print("All resumes indexed into Qdrant successfully")
print("Total candidates:", len(points))
for r in parsed_resumes:
    print(f"  - {r.name}: {len(r.skills)} skills, {r.total_experience_years} years")

Old collection deleted
Collection created with dense + sparse vectors
All resumes indexed into Qdrant successfully
Total candidates: 5
  - Charan Kumar: 13 skills, 0.5 years
  - Rahul Verma: 12 skills, 1.5 years
  - Priya Sharma: 9 skills, 2.0 years
  - Sneha Patel: 9 skills, 3.0 years
  - Arjun Mehta: 10 skills, 2.0 years


In [21]:
from qdrant_client.models import Prefetch, FusionQuery, Fusion

def build_query_from_jd(jd: ParsedJD) -> str:
    # Automatically build search query from JD fields
    query_parts = [
        jd.job_title,
        ' '.join(jd.required_skills),
        ' '.join(jd.preferred_skills),
        ' '.join(jd.responsibilities)
    ]
    query = ' '.join(query_parts)
    return query

def hybrid_search(jd: ParsedJD, top_k: int = 5) -> list:

    # Auto build query from JD
    query_text = build_query_from_jd(jd)

    # Dense vector from query
    dense_vector = embedding_model.encode(
        query_text,
        normalize_embeddings=True
    ).tolist()

    # Sparse vector from query
    sparse_vector = build_sparse_vector(query_text)

    # Hybrid search with RRF fusion in Qdrant
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            Prefetch(query=dense_vector, using="dense", limit=top_k),
            Prefetch(query=sparse_vector, using="sparse", limit=top_k)
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=top_k,
        with_payload=True
    )

    return results.points, query_text

# Test hybrid search
print("Running Hybrid Search from JD...")
print()
candidates, query_used = hybrid_search(parsed_jd, top_k=5)

print("Query built automatically from JD:")
print(query_used[:100], "...")
print()
print("RRF Ranked Candidates:")
print("="*50)
for rank, c in enumerate(candidates, 1):
    print(f"Rank {rank}: {c.payload['name']} | RRF Score: {c.score:.4f}")
    print(f"  Skills: {c.payload['skills'][:4]}")

Running Hybrid Search from JD...

Query built automatically from JD:
Machine Learning Engineer Python Machine Learning Deep Learning NLP Docker Git LangChain HuggingFace ...

RRF Ranked Candidates:
Rank 1: Rahul Verma | RRF Score: 1.0000
  Skills: ['Python', 'NLP', 'Transformers', 'BERT']
Rank 2: Charan Kumar | RRF Score: 0.6667
  Skills: ['Python', 'Machine Learning', 'Deep Learning', 'NLP']
Rank 3: Arjun Mehta | RRF Score: 0.5000
  Skills: ['Python', 'TensorFlow', 'Computer Vision', 'OpenCV']
Rank 4: Priya Sharma | RRF Score: 0.3667
  Skills: ['Java', 'Spring Boot', 'Microservices', 'Kubernetes']
Rank 5: Sneha Patel | RRF Score: 0.3667
  Skills: ['Python', 'Apache Spark', 'Hadoop', 'Airflow']


In [22]:
import numpy as np
from sentence_transformers import CrossEncoder

print("Loading Cross-Encoder...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-Encoder loaded")

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def calculate_skills_match(candidate_skills, required_skills, preferred_skills):
    candidate_lower = [s.lower().strip() for s in candidate_skills]

    required_matched = []
    required_missing = []
    for skill in required_skills:
        skill_lower = skill.lower().strip()
        matched = any(
            skill_lower in c or
            c in skill_lower or
            skill_lower.replace(" ", "") in c.replace(" ", "")
            for c in candidate_lower
        )
        if matched:
            required_matched.append(skill)
        else:
            required_missing.append(skill)

    preferred_matched = []
    preferred_missing = []
    for skill in preferred_skills:
        skill_lower = skill.lower().strip()
        matched = any(skill_lower in c or c in skill_lower for c in candidate_lower)
        if matched:
            preferred_matched.append(skill)
        else:
            preferred_missing.append(skill)

    req_score = len(required_matched) / len(required_skills) * 100 if required_skills else 100
    pref_score = len(preferred_matched) / len(preferred_skills) * 100 if preferred_skills else 100

    return req_score, pref_score, required_matched, required_missing, preferred_matched, preferred_missing

def calculate_experience_score(candidate_exp, min_required):
    if not min_required or min_required == 0:
        return 100
    if not candidate_exp:
        return 0
    if candidate_exp >= min_required:
        return 100
    return round((candidate_exp / min_required) * 100, 2)

def calculate_certification_score(candidate_certs, required_certs):
    if not required_certs:
        return 100
    candidate_lower = [c.lower() for c in candidate_certs]
    matched = sum(1 for cert in required_certs if any(cert.lower() in c for c in candidate_lower))
    return round(matched / len(required_certs) * 100, 2)

def full_ats_pipeline(jd: ParsedJD, top_k: int = 5):

    # Step 1: Hybrid Search with RRF
    candidates, query_used = hybrid_search(jd, top_k)

    # Step 2: Cross-Encoder semantic scoring
    pairs = [[query_used, c.payload["text"]] for c in candidates]
    raw_scores = cross_encoder.predict(pairs)
    semantic_scores = [round(float(sigmoid(s)) * 100, 2) for s in raw_scores]

    # Step 3: Full ATS scoring
    ranked = []
    for candidate, semantic_score in zip(candidates, semantic_scores):

        req_score, pref_score, req_matched, req_missing, pref_matched, pref_missing = calculate_skills_match(
            candidate.payload["skills"],
            jd.required_skills,
            jd.preferred_skills
        )

        exp_score = calculate_experience_score(
            candidate.payload.get("exp_years", 0),
            jd.minimum_experience_years
        )

        cert_score = calculate_certification_score(
            candidate.payload.get("certifications", []),
            jd.certifications_required
        )

        # Hard filter
        if req_score < 40:
            final_ats = round(req_score * 0.40, 2)
            status = "AUTO REJECTED"
        else:
            final_ats = round(
                (req_score      * 0.40) +
                (semantic_score * 0.30) +
                (exp_score      * 0.15) +
                (pref_score     * 0.10) +
                (cert_score     * 0.05),
                2
            )
            if final_ats >= 75:
                status = "SHORTLISTED"
            elif final_ats >= 50:
                status = "REVIEW"
            else:
                status = "REJECTED"

        ranked.append({
            "name": candidate.payload["name"],
            "email": candidate.payload["email"],
            "education": candidate.payload["education"],
            "institution": candidate.payload["institution"],
            "exp_years": candidate.payload.get("exp_years", 0),
            "final_ats_score": final_ats,
            "status": status,
            "required_skills_score": round(req_score, 2),
            "semantic_score": semantic_score,
            "experience_score": exp_score,
            "preferred_skills_score": round(pref_score, 2),
            "certification_score": cert_score,
            "required_matched": req_matched,
            "required_missing": req_missing,
            "preferred_matched": pref_matched,
            "preferred_missing": pref_missing,
            "rrf_score": round(candidate.score, 4)
        })

    ranked = sorted(ranked, key=lambda x: x["final_ats_score"], reverse=True)
    return ranked

# Run full pipeline
print()
print("Running Full ATS Pipeline for:", parsed_jd.job_title, "at", parsed_jd.company)
print("="*60)

final_ranking = full_ats_pipeline(parsed_jd, top_k=5)

print()
print("FINAL ATS RESULTS:")
print("="*60)
for rank, c in enumerate(final_ranking, 1):
    print(f"Rank {rank}: {c['name']}  [{c['status']}]")
    print(f"  Final ATS Score       : {c['final_ats_score']} / 100")
    print(f"  Required Skills (40%) : {c['required_skills_score']}%  matched: {c['required_matched']}")
    print(f"  Semantic Score  (30%) : {c['semantic_score']}")
    print(f"  Experience      (15%) : {c['experience_score']}%  ({c['exp_years']} yrs)")
    print(f"  Preferred Skills(10%) : {c['preferred_skills_score']}%  matched: {c['preferred_matched']}")
    print(f"  Certifications   (5%) : {c['certification_score']}%")
    print(f"  Missing Skills        : {c['required_missing']}")
    print()

Loading Cross-Encoder...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross-Encoder loaded

Running Full ATS Pipeline for: Machine Learning Engineer at Google India

FINAL ATS RESULTS:
Rank 1: Rahul Verma  [SHORTLISTED]
  Final ATS Score       : 82.5 / 100
  Required Skills (40%) : 66.67%  matched: ['Python', 'Machine Learning', 'Deep Learning', 'NLP']
  Semantic Score  (30%) : 99.46
  Experience      (15%) : 100%  (1.5 yrs)
  Preferred Skills(10%) : 60.0%  matched: ['LangChain', 'HuggingFace', 'RAG']
  Certifications   (5%) : 100%
  Missing Skills        : ['Docker', 'Git']

Rank 2: Charan Kumar  [SHORTLISTED]
  Final ATS Score       : 81.75 / 100
  Required Skills (40%) : 100.0%  matched: ['Python', 'Machine Learning', 'Deep Learning', 'NLP', 'Docker', 'Git']
  Semantic Score  (30%) : 97.49
  Experience      (15%) : 50.0%  (0.5 yrs)
  Preferred Skills(10%) : 0.0%  matched: []
  Certifications   (5%) : 100%
  Missing Skills        : []

Rank 3: Arjun Mehta  [AUTO REJECTED]
  Final ATS Score       : 13.33 / 100
  Required Skills (40%) : 33.33%  matched: 